# 01 · Query decomposition

A multi-hop question can require several facts that no single passage contains.
Query decomposition turns the original question into independent search queries,
retrieves evidence for each, and combines the results before answering. The
important question is whether the combined evidence gives the reader what it
needs within a fixed passage budget.

This notebook follows a question from its retrieval plan through pooling and
cited generation. An invented three-fact chain makes those steps visible without
a model service. The recorded MuSiQue results then show how the same method
behaves across two-, three-, and four-hop questions. The experiments vary pooling
and reader budgets to distinguish evidence selection from answer quality.

## 1. Query decomposition

The baseline embeds one question and retrieves once. This arm requests two to
four standalone evidence-search questions, retrieves each plus the original
question, and answers once from five unique pooled passages. It does not discover
an intermediate entity and then rewrite the next query; that is the later IRCoT
step. Planning and pooling both change here, so a score difference cannot be
attributed to splitting alone.

MuSiQue's gold hop graph describes the dependencies an ideal retrieval would
cover. The example below was exported by the pinned, SHA-256 checked loader.
Edges run from prerequisite to dependent step IDs. Gold subquestions and answers
are scorer-only annotations; the held-out answerer sees only the original question.
The displayed example is an answered two-hop case, not a representative sample.

### Fix the comparison before running it

`question → 2–4 independent queries + original query → local retrieval → unique pooled passages → one cited answer`

| Held fixed | Changed by this arm |
| --- | --- |
| Corpus, encoder, final answer template, five-passage reader budget | Planning call, query set, evidence pooling |

This is an independent decomposition: later searches do not receive an answer
discovered by an earlier search. A phrase such as “the composer of the work” is
a retrieval target, not a resolved entity. Notebook 03 studies that distinction.

In [1]:
# ruff: noqa: E402 - resolve checkout imports before the first example
import json
import sys
from pathlib import Path

from IPython.display import Markdown
from IPython.display import display as show


# True runs each model step on MODELS through the proxy and caches the result.
# False replays the reference results from RI #2's release with no model calls.
RUN_EXPENSIVE = False

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "aieng-advanced-rag/aieng").is_dir())
sys.path[:0] = [str(ROOT / "aieng-advanced-rag"), str(ROOT / "implementations/cross_document_synthesis")]
from synthesis_lab.tutorial import read_result, require_compute, show_table


require_compute()

from aieng.advanced_rag.models import choose_models


MODELS = choose_models(show=True)

import numpy as np
from aieng.advanced_rag.corpora.musique import Passage
from aieng.advanced_rag.pipeline.answering import Answerer, DecompositionAnalysis
from synthesis_lab.decomposition import DecompositionAnswerer, ModelDecomposer, pool_rankings
from synthesis_lab.reference import decomposition_example


# The worked example holds MuSiQue question text, plans, and answers, so it ships
# in RI #2's reference release. PER_HOP = 10 matches the reference; another value
# has no reference and runs on MODELS when RUN_EXPENSIVE is True.
PER_HOP = 10
worked = decomposition_example(models=MODELS, live=RUN_EXPENSIVE, per_hop=PER_HOP)
report, example = worked.value["report"], worked.value["example"]

assert example["corpus_revision"] == report["corpus_revision"]
print("Corpus revision:", example["corpus_revision"])
print("Questions SHA-256:", example["questions_sha256"])
print("Passages SHA-256:", example["corpus_sha256"])
print("Questions / passages:", example["corpus_questions"], example["corpus_passages"])
print("Questions by hop count:", example["hop_counts"])
print("Question:", example["question"])
print("Scorer-only hop graph:")
print(json.dumps(example["hop_graph"], indent=2))

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)
decomposition-example reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                      60 calls, 27,404 prompt + 2,153 completion tokens
                      rebuilding with gemini-3.5-flash-lite: about 60 calls, est. $0.01 (set RUN_EXPENSIVE = True to rebuild)
Corpus revision: 5ec05b38deecc3318bb432c69865959c56058990
Questions SHA-256: 98ed4e21d3076532f6388d42320fb809599c63a0d8dffca8ece5e41922be6b46
Passages SHA-256: 73157a03ce3f0b1a5673dd5dc12bb970c24976dbffc688af9eecdd758c97ffcb
Questions / passages: 1000 11656
Questions by hop count: {'2': 518, '3': 316, '4': 166}
Question: When was the person who Messi's goals in 

**Read the graph:** a node identifies a required supporting fact; an edge means
a later fact depends on an earlier answer. MuSiQue has 518 two-hop, 316 three-hop,
and 166 four-hop questions over 11,656 passages. The example graph is displayed
for inspection only. Notice that the answerer call below accepts a question and
passages, never this gold graph.

## 2. A worked example

**Predict first:** the composer, their birthplace, and its country live in three
different passages. Which facts will a single query miss? Can the planner infer
the unknown composer name before reading?

The unit vectors and planning completion below are scripted to expose mechanics.
The final fixture completion checks that all three source facts are present.
The real `DecompositionAnswerer` still validates the plan, retrieves, pools,
builds the common answer prompt, and parses citations.

In [2]:
toy_passages = [
    Passage("toy::0", "Composer", "Mira Vale composed The Glass Harbor."),
    Passage("toy::1", "Other opera", "The Golden Shore is an unrelated opera."),
    Passage("toy::2", "Other place", "Southmere is a different country."),
    Passage("toy::3", "Other composer", "Tara Reed composed a different opera."),
    Passage("toy::4", "Other town", "Birchwell is unrelated to this question."),
    Passage("toy::5", "Birthplace", "Mira Vale was born in Alderwick."),
    Passage("toy::6", "Country", "Alderwick is a town in Northmere."),
    Passage("toy::7", "Premiere", "The Glass Harbor premiered in 1890."),
]
toy_question = "In which country was the composer of The Glass Harbor born?"
toy_parts = [
    "Who composed The Glass Harbor?",
    "Where was the composer of The Glass Harbor born?",
    "Which country contains the birthplace of the composer of The Glass Harbor?",
]
toy_vectors = np.eye(len(toy_passages), dtype=np.float32)
toy_rows = dict(zip([*toy_parts, toy_question], [0, 5, 6, 7], strict=True))
toy_prompts = []


def toy_encode(queries):
    """Return scripted vectors for this fixture, not learned embeddings."""
    return [toy_vectors[toy_rows[query]].tolist() for query in queries]


def toy_complete(prompt):
    """Return scripted completions so the actual answerer can run offline."""
    toy_prompts.append(prompt)
    if prompt.startswith("Split the question"):
        return json.dumps({"subquestions": toy_parts})
    needed = [toy_passages[i].text for i in (0, 5, 6)]
    if all(fact in prompt for fact in needed):
        return json.dumps({"answer": "Northmere", "sources": [1, 2, 3], "not_stated": False})
    return json.dumps({"answer": "", "sources": [], "not_stated": True})


toy_implementation = DecompositionAnswerer(
    toy_passages,
    toy_vectors,
    toy_encode,
    toy_complete,
    toy_complete,
    retrieval_depth=8,
    evidence_budget=5,
)
toy_answerer: Answerer = toy_implementation
toy_result = toy_answerer.answer("toy-example", toy_question)
toy_trace = toy_implementation.traces["toy-example"]
assert toy_result.question == toy_question
assert toy_result.answer.text == "Northmere"
assert set(toy_result.answer.chunk_ids) <= set(toy_trace["generation_ids"])
assert len(toy_prompts) == 2
print(json.dumps(toy_trace["plan"], indent=2))
print("Passages shown to the answer generator:", toy_trace["generation_ids"])
print("Answer:", toy_result.answer.text, "| Cited IDs:", list(toy_result.answer.chunk_ids))

{
  "question": "In which country was the composer of The Glass Harbor born?",
  "steps": [
    {
      "question": "Who composed The Glass Harbor?",
      "scope": null,
      "strategy": "decomposition",
      "parent": "In which country was the composer of The Glass Harbor born?"
    },
    {
      "question": "Where was the composer of The Glass Harbor born?",
      "scope": null,
      "strategy": "decomposition",
      "parent": "In which country was the composer of The Glass Harbor born?"
    },
    {
      "question": "Which country contains the birthplace of the composer of The Glass Harbor?",
      "scope": null,
      "strategy": "decomposition",
      "parent": "In which country was the composer of The Glass Harbor born?"
    }
  ],
  "rewrite_outcome": ""
}
Passages shown to the answer generator: ['toy::0', 'toy::5', 'toy::6', 'toy::7', 'toy::1']
Answer: Northmere | Cited IDs: ['toy::0', 'toy::5', 'toy::6']


### Pool before spending the evidence budget

Compare concatenation with rank-first round-robin pooling. Concatenation spends
all five slots on the first query and misses both later facts in this fixture.
Pooling admits the later queries before truncation. Invalid planner output is
recorded and falls back to the original dense query. A provider failure propagates;
it is not scored as evidence-based abstention.

In [3]:
toy_rankings = toy_trace["step_rankings"]
concatenated = list(dict.fromkeys(pid for ranking in toy_rankings for pid in ranking))[:5]
pooled = pool_rankings(toy_rankings, 5)
print("Concatenate then truncate:", concatenated)
print("Round robin then truncate:", pooled)
assert pooled[:4] == ["toy::0", "toy::5", "toy::6", "toy::7"]
assert {"toy::5", "toy::6"}.isdisjoint(concatenated)
assert pooled != concatenated
bad_planner = ModelDecomposer(lambda question: "malformed response")
fallback = DecompositionAnalysis(bad_planner, max_steps=4).analyse(toy_question)
assert bad_planner.outcome == "invalid-plan"
assert [step.question for step in fallback.retrieval_steps()] == [toy_question]
print("Fallback:", bad_planner.outcome, fallback.retrieval_steps()[0].question)

Concatenate then truncate: ['toy::0', 'toy::1', 'toy::2', 'toy::3', 'toy::4']
Round robin then truncate: ['toy::0', 'toy::5', 'toy::6', 'toy::7', 'toy::1']
Fallback: invalid-plan In which country was the composer of The Glass Harbor born?


**Expected observation:** round-robin pooling keeps the three fact-bearing IDs
before distractors consume the budget. Malformed planning falls back to dense
retrieval; that does not mean the question is unanswerable. Provider errors are
separate operational failures. The synthetic generator is deliberately simple:
it demonstrates evidence availability, not natural-language reasoning ability.

## 3. Recorded results

### First inspect the small integration cohort

The saved teaching run selects ten questions per hop count. Both arms search the
entire corpus and share question IDs, encoder, answer model, prompt, and evidence
budget. Recall@5 is the fraction of annotated supports found; all-supports@5
requires the complete annotated support set. EM/F1 score all selected questions,
including abstentions. This balanced 30-question cohort is an integration example.

Answer EM uses normalized string equality against the best accepted reference;
token F1 measures normalized token overlap against the best alias, averaged per
question. A correct answer with incomplete marked support and an incorrect
answer with complete support are both possible. See notebook 05 for that scoring
exercise and for the distinction between citation identity and entailment.


### Evaluation data and fixed configuration

The table below identifies the saved full-corpus reference used in this section.
The MuSiQue reference searches all 11,656 passages for 1,000 original questions.
The NQ control searches 9,633 passages for its 1,000 questions. Model, encoder,
source revision, prompt identity, and evidence settings travel with the saved
reports. Loading these records makes no model calls and does not recompute the
measurements. New experiments must keep the controls fixed or identify the change.


In [4]:
saved = read_result("evaluation/musique_decomposition.json")
spec = saved["spec"]
assert saved["completed"] and saved["coverage_complete"] and len(saved["records"]) == 1000
show_table(
    ["Recorded setting", "Value"],
    [
        ["Answer model", spec["model"]],
        ["Generation endpoint", spec["base_url"]],
        ["Local encoder", spec["embedding"][0]],
        ["Encoder revision", spec["embedding"][1]],
        ["Corpus revision", spec["provenance"]["revision"]],
        ["Final passage budget", spec["bounds"]["generation_passages"]],
    ],
)
print("Answer-template SHA-256:", spec["answer_prompt_sha256"])
print("Saved configuration SHA-256:", saved["spec_sha256"])

| Recorded setting | Value |
| --- | --- |
| Answer model | Qwen3_6-35B-A3B |
| Generation endpoint | https://proxy.vectorinstitute.ai/v1 |
| Local encoder | BAAI/bge-small-en-v1.5 |
| Encoder revision | 5c38ec7c405ec4b44b94cc5a9bb96e735b38267a |
| Corpus revision | 5ec05b38deecc3318bb432c69865959c56058990 |
| Final passage budget | 5 |

Answer-template SHA-256: 517c503f1380aa40f6d20b1fc0b8f6538bbf367601e79d4bca0d316cea6eea0f
Saved configuration SHA-256: 6c8175bab7bb1e15d697a428c871962ada80456d210ba69e01fb9b535283753f


In [5]:
assert report["completed"]
assert report["corpus_passages"] == 11656
assert report["base_url"] == "https://proxy.vectorinstitute.ai/v1"
assert report["embedding_model"] == "BAAI/bge-small-en-v1.5"
assert report["evidence_budget"] == 5
print("Selection:", report["selection"])
print("Purpose:", report["purpose"])
print("Saved scores below are unchanged by the optional live run.")

Selection: first 10 questions per hop count in pinned file order
Purpose: balanced integration slice; not a population quality estimate
Saved scores below are unchanged by the optional live run.


In [6]:
rows = [
    "| Hops | N | Dense recall@5 | Decomp recall@5 | Dense EM | Decomp EM | Dense F1 | Decomp F1 |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for label in ("2", "3", "4", "overall"):
    dense = report["matched_dense_baseline"][label]
    decomp = report["metrics"][label]
    assert dense["questions"] == decomp["questions"]
    values = [
        dense["retrieval"]["5"]["recall_at_5"],
        decomp["retrieval"]["5"]["recall_at_5"],
        dense["answers"]["exact_match"],
        decomp["answers"]["exact_match"],
        dense["answers"]["token_f1"],
        decomp["answers"]["token_f1"],
    ]
    rows.append(f"| {label} | {decomp['questions']} | " + " | ".join(f"{v:.4f}" for v in values) + " |")
show(Markdown(chr(10).join(rows)))
for arm in ("matched_dense_baseline", "metrics"):
    print(arm, "all-supports@5:", f"{report[arm]['overall']['retrieval']['5']['all_supports_at_5']:.4f}")

| Hops | N | Dense recall@5 | Decomp recall@5 | Dense EM | Decomp EM | Dense F1 | Decomp F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2 | 10 | 0.4500 | 0.5000 | 0.0000 | 0.1000 | 0.0000 | 0.1000 |
| 3 | 10 | 0.3333 | 0.4333 | 0.3000 | 0.3000 | 0.3000 | 0.3667 |
| 4 | 10 | 0.4000 | 0.4250 | 0.2000 | 0.2000 | 0.2400 | 0.2000 |
| overall | 30 | 0.3944 | 0.4528 | 0.1667 | 0.2000 | 0.1800 | 0.2222 |

matched_dense_baseline all-supports@5: 0.0000
metrics all-supports@5: 0.0333


### Trace one result and account for its cost

Compare the model's descriptive queries with the scorer-only gold graph. Inspect
the exact passage IDs the reader saw, not every candidate returned. Read-level
availability and valid citations alone do not prove the answer is entailed.
All planning and generation responses count toward usage; cache hits are separate.

In [7]:
record = next((row for row in report["records"] if row["question_id"] == example["question_id"]), report["records"][0])
print("Question:", record["plan"]["question"])
print("Plan outcome:", record["plan_outcome"])
print(json.dumps(record["plan"]["steps"], indent=2))
print("Generation IDs:", record["generation_ids"])
print("Answer:", record["result"]["answer"])
print("Cited IDs:", record["result"]["chunk_ids"])
if record["question_id"] == example["question_id"]:
    print("Scorer-only gold answer:", example["answer"])
    print(
        "Gold support IDs missing from context:", sorted(set(example["supporting_ids"]) - set(record["generation_ids"]))
    )
outcomes = {
    outcome: sum(row["plan_outcome"] == outcome for row in report["records"])
    for outcome in sorted({row["plan_outcome"] for row in report["records"]})
}
print("All plan outcomes:", outcomes)
print("Abstentions:", sum(row["result"]["abstained"] for row in report["records"]))
print("Usage:", json.dumps(report["usage"]["run"], indent=2))
for row in report["records"]:
    assert row["pooled_ids"] == pool_rankings(row["step_rankings"], 20)
    assert row["generation_ids"] == row["pooled_ids"][:5]
    assert set(row["result"]["chunk_ids"]) <= set(row["generation_ids"])
    assert row["usage"]["calls_missing_usage"] == 0
assert report["usage"]["run"]["total_tokens"] == sum(row["usage"]["total_tokens"] for row in report["records"])

Question: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?
Plan outcome: decomposed
[
  {
    "question": "Who is the person Messi's Copa del Rey goals are compared to?",
    "scope": null,
    "strategy": "decomposition",
    "parent": "When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?"
  },
  {
    "question": "When was that person signed by Barcelona?",
    "scope": null,
    "strategy": "decomposition",
    "parent": "When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?"
  }
]
Generation IDs: ['musique::00016', 'musique::00002', 'musique::00001', 'musique::00019', 'musique::00010']
Answer: June 1982
Cited IDs: ['musique::00002', 'musique::00001']
Scorer-only gold answer: June 1982
Gold support IDs missing from context: []
All plan outcomes: {'decomposed': 28, 'invalid-plan': 2}
Abstentions: 17
Usage: {
  "questions": 30,
  "calls": 58,
  "prompt_tokens": 27404,
 

### Then check the full matched cohort

The committed full comparison covers all 1,000 MuSiQue questions. Unlike the
balanced teaching slice, its hop distribution follows the pinned dataset.
Read each hop group before interpreting the overall mean. The paired difference
uses the same original IDs for both arms. These saved metrics are loaded without
downloads; pinned-annotation replay is separately gated in section 5.

In [8]:
summary = read_result("evaluation/summary.json")
musique = {r["method"]: r for r in summary["reports"] if r["corpus"] == "musique"}
show_table(
    ["Group", "N", "Dense F1", "Decomposition F1", "Dense recall@5", "Decomposition recall@5"],
    [
        [
            group,
            musique["dense"]["scores"][group]["questions"],
            musique["dense"]["scores"][group]["metrics"]["token_f1"],
            musique["decomposition"]["scores"][group]["metrics"]["token_f1"],
            musique["dense"]["scores"][group]["metrics"]["recall_at_5"],
            musique["decomposition"]["scores"][group]["metrics"]["recall_at_5"],
        ]
        for group in ("2-hop", "3-hop", "4-hop", "overall")
    ],
)
comparison = next(r for r in summary["comparisons"] if r["corpus"] == "musique" and r["method"] == "decomposition")
print("Paired F1 difference and 95% interval:", comparison["differences"]["token_f1"])

| Group | N | Dense F1 | Decomposition F1 | Dense recall@5 | Decomposition recall@5 |
| --- | --- | --- | --- | --- | --- |
| 2-hop | 518 | 0.3223 | 0.3120 | 0.6052 | 0.6264 |
| 3-hop | 316 | 0.2458 | 0.2770 | 0.4684 | 0.5443 |
| 4-hop | 166 | 0.1727 | 0.1887 | 0.2681 | 0.3464 |
| overall | 1000 | 0.2733 | 0.2805 | 0.5060 | 0.5540 |

Paired F1 difference and 95% interval: {'mean': 0.007209907980736157, 'low': -0.012703248439726767, 'high': 0.027211134788564538}


The full F1 difference is about +0.0072, with an interval crossing zero.
Improved retrieval recall therefore does not establish an answer-quality gain.
Inspect whether missing supports, pooling, or the reader explains a failure.
Warm-cache calls are not cold execution cost; the original reference usage is
recorded separately in `outputs/reference/execution_usage.json`.

### Passage count and text amount are different budgets

All saved arms show at most five original passages to the final reader. That
controls the source-count budget; passage lengths can still differ. Compare the
actual text amount before attributing every gain to better ranking or reasoning.
Here the two pooling rules both choose five passages but can choose different
amounts of text. Character counts are a simple diagnostic, not tokenizer counts,
unique evidence coverage, or total model-prompt tokens.


In [9]:
by_id = {p.passage_id: p for p in toy_passages}
toy_supports_set = {"toy::0", "toy::5", "toy::6"}
show_table(
    ["Pooling rule", "Passages", "Source-text characters", "Gold supports present / 3"],
    [
        [name, len(ids), sum(len(by_id[pid].text) for pid in ids), len(toy_supports_set.intersection(ids))]
        for name, ids in (("concatenate", concatenated), ("round robin", pooled))
    ],
)

| Pooling rule | Passages | Source-text characters | Gold supports present / 3 |
| --- | --- | --- | --- |
| concatenate | 5 | 185 | 1 |
| round robin | 5 | 175 | 3 |

## 4. Experiments

Change `BUDGETS` and rerun this cell. **Predict first:** with a budget of three,
which pooling rule retains the whole synthetic support chain? Does raising the
budget help both rules equally? This is fixture coverage, not MuSiQue accuracy.

Then change `CANDIDATE_PLAN`: try an empty list, duplicate questions, a `#1`
placeholder, and two valid standalone questions. Observe the explicit planning
outcome. Keep the evidence budget fixed when comparing alternative splitters.

In [10]:
BUDGETS = [1, 3, 5]
toy_supports = {"toy::0", "toy::5", "toy::6"}
for budget in BUDGETS:
    first_query = list(dict.fromkeys(pid for ranking in toy_rankings for pid in ranking))[:budget]
    round_robin = pool_rankings(toy_rankings, budget)
    print(
        f"Budget {budget}: concatenation {len(toy_supports.intersection(first_query))}/3 facts; "
        f"round robin {len(toy_supports.intersection(round_robin))}/3 facts"
    )
CANDIDATE_PLAN = {"subquestions": ["Who composed The Glass Harbor?", "When was #1 born?"]}
candidate = ModelDecomposer(lambda question: json.dumps(CANDIDATE_PLAN))
candidate_parts = candidate(toy_question)
print("Candidate outcome:", candidate.outcome, "| Accepted queries:", candidate_parts)

Budget 1: concatenation 1/3 facts; round robin 1/3 facts
Budget 3: concatenation 1/3 facts; round robin 3/3 facts
Budget 5: concatenation 1/3 facts; round robin 3/3 facts
Candidate outcome: invalid-plan | Accepted queries: []


### Measure the reader consequence, not just coverage

Now the same budget control runs the actual answering adapter. **Predict first:**
which budget lets the source-checking fixture answer rather than abstain?
Budgets below five are teaching ablations, not the configuration of the saved
MuSiQue benchmark. Keep the restored planner prompt unchanged for the reference.

In [11]:
budget_rows = []
for budget in BUDGETS:
    variant = DecompositionAnswerer(
        toy_passages,
        toy_vectors,
        toy_encode,
        toy_complete,
        toy_complete,
        retrieval_depth=8,
        evidence_budget=budget,
    )
    answer = variant.answer(f"budget-{budget}", toy_question)
    shown = variant.traces[f"budget-{budget}"]["generation_ids"]
    budget_rows.append([budget, len(toy_supports.intersection(shown)), answer.answer.text or "(abstain)", shown])
show_table(["Reader budget", "Supports present / 3", "Fixture answer", "Generation IDs"], budget_rows)

| Reader budget | Supports present / 3 | Fixture answer | Generation IDs |
| --- | --- | --- | --- |
| 1 | 1 | Not stated in the provided sources. | ['toy::0'] |
| 3 | 3 | Northmere | ['toy::0', 'toy::5', 'toy::6'] |
| 5 | 3 | Northmere | ['toy::0', 'toy::5', 'toy::6', 'toy::7', 'toy::1'] |

## 5. Run the worked example on your model

The worked example in section 3 comes from `decomposition_example`, one step behind
`load_or_run`. With `RUN_EXPENSIVE = False` it replays the reference built with
`Qwen3_6-35B-A3B` from RI #2's release and prints what a rebuild on your model
would cost. With `RUN_EXPENSIVE = True` it plans and answers the same 30 questions
on `MODELS` and caches the result, which later runs replay. Change `PER_HOP` in the
first code cell for another slice; zero selects all 1,000.

The switch below also gates corpus replay, which checks source digests and scores
and may download the pinned files, but makes no model calls.

Live chat uses the shared model roles and Vector credential resolver. The
default family is Google; use `ADVANCED_RAG_MODEL_FAMILY=onprem` for Qwen
generation and extraction. Embeddings stay local. Set
`ADVANCED_RAG_SPEND_LIMIT_USD` before execution to limit spending. See the
[provider settings](GUIDE.md#models-credentials-and-spend). Recorded results
retain their original models; fresh comparisons require matched model roles.

In [12]:
RUN_CORPUS = RUN_EXPENSIVE

### Optional pinned-corpus replay

With `RUN_EXPENSIVE = True`, this cell loads both digest-checked files and
recomputes scores and the example graph without embeddings or proxy calls. It validates the live report when present, otherwise the saved report.
Gold annotations enter only after answering.

In [13]:
if RUN_CORPUS:
    from dataclasses import asdict

    from aieng.advanced_rag.corpora.musique import fetch_musique, load_musique
    from synthesis_lab.dense_baseline import score_by_hops

    passages, questions = load_musique(fetch_musique(dest=ROOT / "Arm Store/musique-dense/corpus"))
    by_id = {q.question_id: q for q in questions}
    active_report = report
    selected = [by_id[row["question_id"]] for row in active_report["records"]]
    runs = {row["question_id"]: row["pooled_ids"] for row in active_report["records"]}
    predictions = {
        row["question_id"]: "" if row["result"]["abstained"] else row["result"]["answer"]
        for row in active_report["records"]
    }
    assert score_by_hops(selected, runs, [5, 10, 20], predictions) == active_report["metrics"]
    assert json.loads(json.dumps(asdict(by_id[example["question_id"]].hop_graph))) == example["hop_graph"]
    print("Verified pinned corpus, hop graph, and all recorded hop-count scores.")
else:
    print("Pinned-corpus replay skipped; set RUN_EXPENSIVE = True to enable it.")

Pinned-corpus replay skipped; set RUN_EXPENSIVE = True to enable it.


## 6. Interpretation

You have separated planning, evidence selection, and final answering. A valid
plan is not a guarantee that its descriptive queries retrieve the missing entity.
A valid citation is not proof of entailment. A larger candidate pool can still
fail to put the entire chain in the reader's five passages.

Before moving on, write down:

1. One failure caused by retrieval, one by pooling, and one possible reader failure.
2. An ablation that changes pooling while fixing the planner and reader budget.
3. Why the 30-question slice and full-corpus result support different claims.

Next, [notebook 02](02_agentic_retrieval.ipynb) exposes search/read tools, and
[notebook 03](03_ircot.ipynb) makes later queries depend on observed evidence.

Keep gold answers and hop graphs with the scorer. Local pinned embeddings and
the Vector proxy are used for live RI #2 experiments. Source IDs establish
provenance; they do not by themselves establish factual entailment.

See the [participant guide](GUIDE.md), [corpus/provenance record](CORPUS.md),
[experiment contracts](docs/EXPERIMENTS.md), and [full matched comparison](docs/COMPARISONS.md).